# Adaptive Semantic Shot Clustering for Stable Keyframe Extraction — v2
### AIC video-retrieval keyframe pipeline — Kaggle T4×2 — encoder: **PE-Core-G14-448**

**Method (đã cải tiến theo `keyframe_pipeline_report.md`):**
`Shot detection (TransNetV2 → PySceneDetect → histogram) → tín hiệu nổi bật S(t) [thị giác + âm thanh] → lấy mẫu thích nghi (nền ∪ sự kiện) → stability scoring → PE-Core-G14-448 embedding → DBSCAN semantic clustering per shot → representative+stable selection → dedup theo coverage suy giảm thời gian → AIC output format`

**Điểm mới so với bản cũ:**
- **Sửa 3 lỗi shot detection** (off-by-one TransNetV2, gộp cảnh đầu khi quá ngắn, histogram đọc đúng config).
- **Tín hiệu nổi bật S(t)** = `ReLU(α·Δ_thị_giác + β·Δ_âm_thanh − ε)` (âm thanh trích qua **ffmpeg**, tự fallback `β=0`).
- **Lấy mẫu thích nghi**: mẫu nền (lưới an toàn) ∪ mẫu sự kiện quanh đỉnh S(t).
- **Brightness Gaussian** (hết phạt oan cảnh sáng), **DBSCAN noise-update fix**.
- **Dedup mới**: `Phủ(f,K)=max_k[cos·exp(−|Δt|/τ)]`, giữ nếu `< θ` → bảo toàn các lần xuất hiện độc lập cách xa thời gian.
- **🆕 Chọn loại video để xử lý** qua `CONFIG["video_groups"]` (vd `["K01","K02","L21","L22"]`).

**Encoder = PE-Core-G14-448** (Meta Perception Encoder, CLIP-dim 1280, 448px). Embedding keyframe được **lưu `.npy`** để tái dùng cho bước index. Fallback chain: PE-Core → SigLIP2 → handcrafted.

**Output (đúng format chuẩn AIC):**
```
/kaggle/working/keyframes/<video_id>/<n>.jpg          # n = thứ tự keyframe, 1-based (001, 002, ...)
/kaggle/working/map-keyframes/<video_id>.csv          # n, frame_idx, pts_time, fps, shot_id, ...
/kaggle/working/embeddings/<video_id>.npy             # (N, 1280) float16, hàng i ↔ n=i+1 (PE-Core)
```
`frame_idx` = frame GỐC trong video → sinh file nộp `video_id, frame_idx`. Khóa vector DB: `<video_id>#<n>` (vd `K20_V001#003`).

**Cách dùng nhanh**
1. Attach dataset video (K01–K20, L21–L30...) → Settings: GPU **T4×2**, Internet **ON** (tải PE-Core + TransNetV2).
2. **Chọn loại video:** đặt `CONFIG["video_groups"]=["K01","K02"]` (rỗng = tất cả). Chạy `list_groups()` để xem các nhóm có trong input.
3. Run All. Cell *smoke test* chạy 1 video; OK thì cell *batch* chạy toàn bộ (đã lọc theo nhóm).
4. Chia ca bằng `RANGE_START`/`RANGE_END` (Kaggle ~9–12h/session). Pipeline **tự resume** (bỏ qua video đã có map-keyframes).
5. PE-Core-G14-448 nặng → cần nhanh hơn: đổi `CONFIG["pe_config"]="PE-Core-L14-336"`. Muốn nhanh hơn nữa: `use_saliency=False` hoặc `saliency_audio=False`.

## 1. Cài đặt dependencies

In [ ]:
import os, sys, subprocess

def pipi(args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

# transformers mới (cho SigLIP2 fallback) + tiện ích
pipi(["-U", "transformers", "accelerate"])
pipi(["scenedetect[opencv]", "imagehash"])
# TransNetV2 (PyTorch, bundle sẵn weights) — optional
pipi(["transnetv2-pytorch"])

# ---- PE-Core (Perception Encoder, Meta) ----
PE_REPO = "/kaggle/working/perception_models"
if not os.path.isdir(PE_REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/facebookresearch/perception_models.git", PE_REPO], check=False)
# editable install KHÔNG kéo theo deps -> giữ nguyên torch/cuda của Kaggle
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", PE_REPO, "--no-deps"], check=False)
pipi(["einops", "timm", "ftfy", "regex", "iopath"])   # deps nhẹ mà PE thực sự cần
if PE_REPO not in sys.path:
    sys.path.insert(0, PE_REPO)
try:
    import core.vision_encoder.pe as _pe
    print("[ok] PE-Core configs:", _pe.CLIP.available_configs())
except Exception as e:
    print(f"[warn] PE chưa import được ({type(e).__name__}: {e}). "
          f"Notebook sẽ tự fallback SigLIP2. Nếu lỗi do attention, thử: pipi(['xformers','--no-deps']).")

# Kiểm tra core libs
for m in ["cv2", "torch", "numpy", "pandas", "sklearn", "PIL", "imagehash", "tqdm"]:
    try:
        __import__(m); print(f"[ok] {m}")
    except Exception as e:
        print(f"[MISSING] {m}: {e}")


## 2. Imports & CONFIG

In [ ]:
import os, glob, json, time, math, traceback, warnings, subprocess, shutil, threading, re
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import imagehash
from sklearn.cluster import DBSCAN
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)  # tránh OpenCV nuốt hết CPU, để dành cho decode song song

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()
print(f"DEVICE={DEVICE} | n_gpu={N_GPU}")
if DEVICE == "cuda":
    for i in range(N_GPU):
        print("  -", torch.cuda.get_device_name(i))

CONFIG = {
    # ---- input/output ----
    "input_roots": ["/kaggle/input"],          # quét đệ quy tất cả dataset đã attach
    "video_exts": (".mp4", ".mkv", ".avi", ".mov", ".webm"),
    "output_dir": "/kaggle/working",            # tạo keyframes/ và map-keyframes/ ở đây

    # ---- CHỌN LOẠI VIDEO (mới) ----
    # Lọc theo prefix của video_id (vd "K20_V001" -> nhóm "K20"). Khớp không phân biệt hoa thường.
    # [] hoặc None = chạy TẤT CẢ. Ví dụ: ["K01", "K02", "L21", "L22"] hoặc ["K0", "L2"].
    "video_groups": [],

    # ---- chia ca chạy (resume-friendly) ----
    "RANGE_START": 0,                           # index video bắt đầu (sau khi sort + lọc nhóm)
    "RANGE_END": None,                          # None = hết; vd 200 để chạy 200 video đầu
    "skip_existing": True,                      # bỏ qua video đã có map-keyframes csv

    # ---- shot detection ----
    "shot_method": "auto",                      # auto | transnetv2 | scenedetect | histogram
    "scenedetect_threshold": 27.0,
    "hist_threshold": 0.45,                     # cho fallback histogram (Bhattacharyya)
    "shot_detection_sample_fps": 2.0,           # tốc độ lấy mẫu cho histogram fallback (đọc từ config, hết hardcode)
    "min_shot_sec": 0.6,                        # gộp shot ngắn hơn ngưỡng này

    # ---- tín hiệu nổi bật S(t) — Module 2A (mới) ----
    "use_saliency": True,                       # bật adaptive event sampling theo S(t)
    "saliency_audio": True,                     # dùng cả tín hiệu âm thanh (ffmpeg). Lỗi -> tự hạ beta=0
    "saliency_sample_fps": 6.0,                 # lưới thời gian của S(t) (thị giác)
    "saliency_alpha": 0.60,                     # trọng số thay đổi thị giác
    "saliency_beta": 0.40,                      # trọng số thay đổi âm thanh
    "saliency_epsilon": 0.15,                   # ngưỡng nhiễu nền (ReLU)
    "saliency_peak_radius_sec": 0.25,           # δ: bán kính cửa sổ quanh đỉnh sự kiện
    "audio_sr": 16000,                          # sample rate khi trích audio
    "audio_rms_window_ms": 100,                 # cửa sổ RMS

    # ---- candidate sampling ----
    "candidate_fps": 2.0,                       # mẫu nền cho shot dài (>20s)
    "max_candidates_per_shot": 40,
    "boundary_margin_frac": 0.06,               # né frame sát ranh giới shot
    # Khi tổng ứng viên > max_candidates_per_shot:
    #   "balanced"    -> giữ lưới nền tối thiểu trước (lưới an toàn), còn lại ưu tiên sự kiện theo S cao nhất
    #   "event_first" -> đúng report: ưu tiên sự kiện, nền chỉ lấp phần dư (có thể bỏ sót vùng tĩnh)
    "candidate_priority": "balanced",
    "min_background_reserve_frac": 0.5,         # phần cap dành tối thiểu cho mẫu nền ở shot dài (balanced)

    # ---- embedding ----
    "embedder_backend": "pe",                   # pe | siglip2 | handcrafted (pe = mạnh nhất)
    "pe_config": "PE-Core-G14-448",             # hoặc "PE-Core-L14-336" (nhẹ/nhanh hơn nhiều)
    "pe_repo_dir": "/kaggle/working/perception_models",
    "pe_multi_gpu": True,                        # 2 instance / 2 GPU (KHÔNG dùng DataParallel); tự hạ 1 GPU nếu lỗi
    "model_name": "google/siglip2-base-patch16-224",   # dùng khi fallback siglip2
    "local_model_path": None,                   # offline: trỏ tới checkpoint/model local
    "embed_batch_size": 8,                       # *mỗi GPU*; PE-G14-448 nặng -> để nhỏ (L14 có thể tăng)
    "save_max_side": 720,                       # downscale frame lưu/encode (giữ keyframe nhẹ)
    "save_embeddings": True,                     # lưu embedding keyframe -> tái dùng cho index
    "embeddings_dir": "embeddings",

    # ---- DBSCAN + selection ----
    "dbscan_eps": 0.18,
    "dbscan_min_samples": 2,
    "w_representativeness": 0.70,
    "w_stability": 0.30,
    "noise_keep_min_stability": 0.62,           # giữ frame noise nếu đủ đẹp & shot dài

    # ---- stability scoring ----
    "w_sharpness": 0.5, "w_brightness": 0.3, "w_contrast": 0.2,
    "brightness_sigma": 0.3,                    # độ rộng Gaussian quanh mức sáng lý tưởng 0.5

    # ---- dedup (thiết kế lại: coverage suy giảm thời gian) ----
    "dedup_tau_sec": 60.0,                      # τ: hằng số thời gian suy giảm
    "dedup_coverage_threshold": 0.85,           # θ: loại nếu coverage >= θ
    "dedup_phash_prefilter": 18,                # pHash > ngưỡng -> chắc chắn khác -> bỏ qua cosine (chỉ tăng tốc)

    # ---- output ----
    "jpg_quality": 90,
}

OUT = Path(CONFIG["output_dir"])
(OUT / "keyframes").mkdir(parents=True, exist_ok=True)
(OUT / "map-keyframes").mkdir(parents=True, exist_ok=True)
(OUT / CONFIG["embeddings_dir"]).mkdir(parents=True, exist_ok=True)

def log(*a):
    print("[kf]", *a, flush=True)

# ffmpeg có sẵn trên Kaggle? Nếu không -> tự tắt nhánh âm thanh của saliency.
HAS_FFMPEG = shutil.which("ffmpeg") is not None
if CONFIG["saliency_audio"] and not HAS_FFMPEG:
    log("ffmpeg không tìm thấy -> tắt nhánh âm thanh của saliency (beta=0).")
    CONFIG["saliency_audio"] = False


In [ ]:
# === Fix giải mã AV1 trên Kaggle ===
# Một phần video AIC mã hoá AV1. Kaggle/T4 KHÔNG hỗ trợ giải mã AV1 bằng GPU -> OpenCV báo
# "Your platform doesn't support hardware accelerated AV1 decoding / Failed to get pixel format /
# Get current frame error" và có thể KHÔNG đọc được frame (video đó ra 0 keyframe trong batch).
# -> Ép ffmpeg của OpenCV dùng giải mã SOFTWARE (chậm hơn chút nhưng chạy được mọi codec).
# (TransNetV2 dùng ffmpeg riêng nên không bị ảnh hưởng; fix này chỉ cho các cv2.VideoCapture.)
os.environ["OPENCV_FFMPEG_CAPTURE_OPTIONS"] = "hwaccel;none"
print("OPENCV_FFMPEG_CAPTURE_OPTIONS =", os.environ.get("OPENCV_FFMPEG_CAPTURE_OPTIONS"))

def video_codec(path):
    """Đọc codec video qua ffprobe (vd 'h264', 'av1', 'hevc'). Lỗi -> '?'."""
    try:
        r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                            "-show_entries", "stream=codec_name", "-of", "csv=p=0", str(path)],
                           stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        return (r.stdout.strip() or "?")
    except Exception:
        return "?"

def codec_report(paths=None, limit=None):
    """Thống kê codec các video (mặc định: theo bộ lọc hiện tại). limit=N để chỉ kiểm N video đầu cho nhanh."""
    from collections import Counter
    paths = paths if paths is not None else discover_videos()
    if limit: paths = paths[:limit]
    cnt = Counter()
    av1 = []
    for p in tqdm(paths, desc="ffprobe"):
        cd = video_codec(p); cnt[cd] += 1
        if cd == "av1": av1.append(Path(p).stem)
    print("Codec:", dict(cnt))
    if av1:
        print(f"AV1 ({len(av1)}):", av1[:20], "..." if len(av1) > 20 else "")
    return cnt, av1


## 3. Data classes & tiện ích timestamp

In [ ]:
@dataclass
class Shot:
    shot_id: int
    start_frame: int
    end_frame: int          # inclusive
    fps: float
    @property
    def start_sec(self): return self.start_frame / self.fps
    @property
    def end_sec(self):   return self.end_frame / self.fps
    @property
    def duration_sec(self): return (self.end_frame - self.start_frame + 1) / self.fps

@dataclass
class Candidate:
    shot_id: int
    frame_idx: int
    pts_time: float
    image_rgb: np.ndarray = field(repr=False)          # uint8 RGB (đã downscale)
    sharpness: float = 0.0
    brightness: float = 0.0
    contrast: float = 0.0
    stability: float = 0.0
    embedding: Optional[np.ndarray] = field(default=None, repr=False)
    phash: Optional[object] = field(default=None, repr=False)
    # gán khi được chọn
    cluster_id: int = -1
    representativeness: float = 0.0
    final_score: float = 0.0
    selection_reason: str = ""

def sec_to_hms(t: float) -> str:
    h = int(t // 3600); m = int((t % 3600) // 60)
    s = t % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}"


## 4. Video IO

In [ ]:
def probe_video(path: str):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()
    if not fps or fps <= 0 or math.isnan(fps): fps = 25.0
    return fps, total, w, h

def _resize_max_side(img_bgr, max_side):
    h, w = img_bgr.shape[:2]
    m = max(h, w)
    if m <= max_side: return img_bgr
    sc = max_side / m
    return cv2.resize(img_bgr, (int(round(w*sc)), int(round(h*sc))), interpolation=cv2.INTER_AREA)

def read_frames_at_indices(path: str, indices, max_side: int):
    # Decode tuần tự 1 lần, chỉ retrieve các frame mục tiêu (grab() rẻ hơn retrieve()).
    # Trả về dict {frame_idx: rgb_uint8 đã downscale}.
    if not indices: return {}
    want = sorted(set(int(i) for i in indices))
    want_set = set(want); last = want[-1]
    out = {}
    cap = cv2.VideoCapture(path)
    i = 0
    while i <= last:
        if not cap.grab():
            break
        if i in want_set:
            ok, frame = cap.retrieve()
            if ok and frame is not None:
                frame = _resize_max_side(frame, max_side)
                out[i] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        i += 1
    cap.release()
    return out


## 5. Shot detection — hierarchy `TransNetV2 → PySceneDetect → histogram`
Bất kỳ lỗi nào ở tầng trên đều tự rớt xuống tầng dưới. `shot_method="auto"` (mặc định) ưu tiên TransNetV2.

In [ ]:
# ---------- TransNetV2 (optional, mạnh nhất) ----------
_TNV2_MODEL = None
def _load_transnetv2():
    global _TNV2_MODEL
    if _TNV2_MODEL is not None: return _TNV2_MODEL
    try:
        from transnetv2_pytorch import TransNetV2
        m = TransNetV2()
        m.eval()
        if DEVICE == "cuda": m = m.to(DEVICE)
        _TNV2_MODEL = m
        log("TransNetV2 loaded.")
    except Exception as e:
        log(f"TransNetV2 unavailable -> fallback. ({type(e).__name__}: {e})")
        _TNV2_MODEL = False
    return _TNV2_MODEL

def _predictions_to_scenes(preds, threshold=0.5):
    # p̂(t)=1 -> khung CHUYỂN CẢNH. Một cảnh = khoảng liên tục các t có p̂(t)=0.
    # FIX (off-by-one): ranh giới cảnh = khung cuối THUỘC nội dung cảnh = (khung chuyển cảnh - 1),
    # không gộp khung chuyển cảnh (thường nhòe) vào cảnh trước.
    p = (np.asarray(preds).reshape(-1) > threshold).astype(np.uint8)
    scenes, t_prev, start, i = [], 0, 0, 0
    t = 0
    for i, t in enumerate(p):
        if t_prev == 1 and t == 0: start = i
        if t_prev == 0 and t == 1 and i != 0: scenes.append([start, i - 1])
        t_prev = t
    if len(p) and t == 0: scenes.append([start, len(p) - 1])
    return scenes

def detect_shots_transnetv2(path, fps, total):
    m = _load_transnetv2()
    if not m: return None
    try:
        with torch.no_grad():
            _, single_pred, _ = m.predict_video(path)
        if hasattr(single_pred, "detach"):
            single_pred = single_pred.detach().cpu().numpy()
        pairs = _predictions_to_scenes(single_pred)
        if not pairs: return None
        return [(int(s), int(e)) for s, e in pairs]
    except Exception as e:
        log(f"TransNetV2 predict failed -> fallback. ({type(e).__name__}: {e})")
        return None

# ---------- PySceneDetect ----------
def detect_shots_scenedetect(path, fps, total):
    try:
        from scenedetect import open_video, SceneManager
        from scenedetect.detectors import ContentDetector
        video = open_video(path)
        sm = SceneManager()
        sm.add_detector(ContentDetector(threshold=CONFIG["scenedetect_threshold"]))
        sm.detect_scenes(video, show_progress=False)
        scenes = sm.get_scene_list()
        if not scenes:
            return [(0, max(0, total - 1))]
        return [(s.get_frames(), max(s.get_frames(), e.get_frames() - 1)) for s, e in scenes]
    except Exception as e:
        log(f"PySceneDetect failed -> histogram. ({type(e).__name__}: {e})")
        return None

# ---------- OpenCV histogram (luôn chạy được) ----------
def detect_shots_histogram(path, fps, total):
    # FIX: đọc tốc độ lấy mẫu từ CONFIG thay vì hardcode 4.0
    sample_fps = CONFIG["shot_detection_sample_fps"]
    step = max(1, int(round(fps / max(0.1, sample_fps))))
    cap = cv2.VideoCapture(path)
    prev_hist, boundaries, i = None, [0], 0
    while True:
        ok = cap.grab()
        if not ok: break
        if i % step == 0:
            ok2, frame = cap.retrieve()
            if ok2 and frame is not None:
                hsv = cv2.cvtColor(_resize_max_side(frame, 320), cv2.COLOR_BGR2HSV)
                hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
                cv2.normalize(hist, hist)
                if prev_hist is not None:
                    d = cv2.compareHist(prev_hist, hist, cv2.HISTCMP_BHATTACHARYYA)
                    if d > CONFIG["hist_threshold"]:
                        boundaries.append(i)
                prev_hist = hist
        i += 1
    cap.release()
    end = max(i - 1, 0)
    boundaries.append(end + 1)
    boundaries = sorted(set(boundaries))
    pairs = []
    for a, b in zip(boundaries[:-1], boundaries[1:]):
        pairs.append((a, max(a, b - 1)))
    return pairs or [(0, end)]

def _merge_short(pairs, fps, total):
    # Gộp shot ngắn hơn min_shot_sec vào shot trước; clamp về [0, total-1].
    min_f = max(1, int(CONFIG["min_shot_sec"] * fps))
    merged = []
    for s, e in pairs:
        s = max(0, s); e = min(e, total - 1) if total > 0 else e
        if e < s: e = s
        if merged and (e - s + 1) < min_f:
            merged[-1] = (merged[-1][0], e)
        else:
            merged.append((s, e))
    # FIX: cảnh ĐẦU TIÊN không có cảnh trước để gộp -> gộp vào cảnh kế (mở rộng cảnh kế về trái).
    if len(merged) >= 2 and (merged[0][1] - merged[0][0] + 1) < min_f:
        merged[1] = (merged[0][0], merged[1][1])
        merged.pop(0)
    return merged

def detect_shots(path, fps, total):
    method = CONFIG["shot_method"]
    order = {"auto": ["transnetv2", "scenedetect", "histogram"],
             "transnetv2": ["transnetv2", "scenedetect", "histogram"],
             "scenedetect": ["scenedetect", "histogram"],
             "histogram": ["histogram"]}[method]
    pairs, used = None, None
    for m in order:
        pairs = {"transnetv2": detect_shots_transnetv2,
                 "scenedetect": detect_shots_scenedetect,
                 "histogram": detect_shots_histogram}[m](path, fps, total)
        if pairs:
            used = m; break
    if not pairs:
        pairs, used = [(0, max(0, total - 1))], "single"
    pairs = _merge_short(pairs, fps, total)
    shots = [Shot(sid, s, e, fps) for sid, (s, e) in enumerate(pairs)]
    return shots, used


## 5b. Module 2A — Tín hiệu nổi bật S(t) (thị giác + âm thanh)
`S(t) = ReLU(α·Δ_thị_giác + β·Δ_âm_thanh − ε)`, chuẩn hoá **per-shot theo phân vị 95**.
- **Thị giác:** 1 lượt decode low-res 64×64, khoảng cách Bhattacharyya giữa histogram HSV hai frame liên tiếp ở `saliency_sample_fps`.
- **Âm thanh:** trích PCM 16kHz mono bằng **ffmpeg** (đọc thẳng vào numpy, không thêm pip dep), năng lượng RMS cửa sổ 100ms, lấy `|d/dt RMS|`, nội suy về lưới thị giác.
- **Fallback:** thiếu ffmpeg / video không audio / lỗi codec → `β=0`, pipeline vẫn chạy bằng thị giác. Tắt hẳn bằng `use_saliency=False`.

Tín hiệu này hướng dẫn bước **lấy mẫu thích nghi** (5c): lấy mẫu dày quanh các đỉnh S(t) — nơi nhiều khả năng có sự kiện.

In [ ]:
# ---------- Δ âm thanh: trích audio 16kHz mono qua ffmpeg, RMS theo cửa sổ ----------
def _extract_audio_rms(path):
    """Trả (times_sec, rms) hoặc None nếu tắt audio / không có ffmpeg / video không audio / lỗi codec."""
    if not CONFIG["saliency_audio"] or not HAS_FFMPEG:
        return None
    sr = CONFIG["audio_sr"]
    win = max(1, int(sr * CONFIG["audio_rms_window_ms"] / 1000))
    try:
        cmd = ["ffmpeg", "-v", "quiet", "-i", path, "-vn",
               "-ac", "1", "-ar", str(sr), "-f", "s16le", "-"]
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        if proc.returncode != 0 or not proc.stdout:
            return None
        audio = np.frombuffer(proc.stdout, np.int16).astype(np.float32) / 32768.0
        if audio.size < win:
            return None
        n = audio.size // win
        frames = audio[:n * win].reshape(n, win)
        rms = np.sqrt((frames ** 2).mean(axis=1) + 1e-9)
        times = (np.arange(n) + 0.5) * (win / sr)
        return times, rms
    except Exception as e:
        log(f"audio extract failed ({type(e).__name__}: {e}) -> beta=0")
        return None

# ---------- Tín hiệu nổi bật toàn video (chưa chuẩn hoá; chuẩn hoá per-shot ở shot_saliency) ----------
def compute_saliency(path, fps, total):
    """1 lượt decode low-res 64x64 -> Δ thị giác (Bhattacharyya HSV) trên lưới saliency_sample_fps.
       + Δ âm thanh = |d/dt RMS| nội suy về cùng lưới. Trả (grid_frame_idx, dvis, daud|None)."""
    if not CONFIG["use_saliency"]:
        return np.array([], int), np.array([], np.float32), None
    step = max(1, int(round(fps / max(0.1, CONFIG["saliency_sample_fps"]))))
    cap = cv2.VideoCapture(path)
    prev, deltas, idxs, i = None, [], [], 0
    while True:
        if not cap.grab(): break
        if i % step == 0:
            ok, frame = cap.retrieve()
            if ok and frame is not None:
                small = cv2.resize(frame, (64, 64), interpolation=cv2.INTER_AREA)
                hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
                hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
                cv2.normalize(hist, hist)
                d = 0.0 if prev is None else float(cv2.compareHist(prev, hist, cv2.HISTCMP_BHATTACHARYYA))
                deltas.append(d); idxs.append(i); prev = hist
        i += 1
    cap.release()
    if not idxs:
        return np.array([], int), np.array([], np.float32), None
    grid = np.asarray(idxs, int)
    dvis = np.asarray(deltas, np.float32)

    daud = None
    res = _extract_audio_rms(path)
    if res is not None:
        atimes, arms = res
        d_arms = np.abs(np.gradient(arms)) if arms.size > 1 else np.zeros_like(arms)
        grid_times = grid / max(1e-6, fps)
        daud = np.interp(grid_times, atimes, d_arms, left=0.0, right=0.0).astype(np.float32)
    return grid, dvis, daud

def _norm95(x):
    if x.size == 0: return x
    p = np.percentile(x, 95)
    return (x / p) if p > 1e-9 else np.zeros_like(x)

def shot_saliency(shot, grid, dvis, daud):
    """S(t)=ReLU(alpha*Δvis + beta*Δaud - eps) cho riêng 1 shot; chuẩn hoá per-shot theo phân vị 95."""
    if grid.size == 0:
        return np.array([], int), np.array([], np.float32)
    mask = (grid >= shot.start_frame) & (grid <= shot.end_frame)
    g = grid[mask]
    if g.size == 0:
        return g, np.array([], np.float32)
    vis = _norm95(dvis[mask])
    alpha, beta, eps = CONFIG["saliency_alpha"], CONFIG["saliency_beta"], CONFIG["saliency_epsilon"]
    if daud is not None:
        aud = _norm95(daud[mask])
    else:
        beta, aud = 0.0, np.zeros_like(vis)
    S = np.maximum(alpha * vis + beta * aud - eps, 0.0).astype(np.float32)
    return g, S

def _local_peaks(g, S):
    """Đỉnh cục bộ: S>0 và >= cả hai lân cận, lớn hơn STRICT ít nhất một bên.
       Bắt được cả spike đơn lẫn đỉnh phẳng (plateau bắt cạnh lên/xuống). Biên = lân cận khuyết coi như -1."""
    peaks, n = [], len(S)
    for k in range(n):
        if S[k] <= 0: continue
        l = S[k - 1] if k > 0 else -1.0
        r = S[k + 1] if k < n - 1 else -1.0
        if S[k] >= l and S[k] >= r and (S[k] > l or S[k] > r):
            peaks.append((int(g[k]), float(S[k])))
    return peaks


## 6. Lấy mẫu thích nghi — Module 2B (`Mẫu_nền ∪ Mẫu_sự_kiện`)
- **Mẫu nền (lưới an toàn):** lấy mẫu theo phân số cố định/`candidate_fps` như cũ → không bỏ sót khoảng thời gian nào dù `S(t)=0`.
- **Mẫu sự kiện:** thêm frame trong cửa sổ `±δ` quanh các **đỉnh cục bộ của S(t)** → lấy mẫu dày đúng nơi có sự kiện.
- **Khi vượt `max_candidates_per_shot`:**
  - `candidate_priority="balanced"` (mặc định): giữ một bộ mẫu nền tối thiểu (trải đều) làm lưới an toàn, phần còn lại ưu tiên đỉnh S(t) cao nhất.
  - `candidate_priority="event_first"`: đúng report — ưu tiên sự kiện, nền chỉ lấp phần dư.

In [ ]:
def _even_subsample(lst, k):
    """Lấy k phần tử trải đều từ lst (giữ thứ tự). k>=len -> trả nguyên."""
    lst = list(lst)
    if k >= len(lst) or len(lst) == 0:
        return lst
    sel = np.linspace(0, len(lst) - 1, k).round().astype(int)
    return [lst[int(t)] for t in sel]

def candidate_indices(shot: Shot, sal_grid=None, sal_S=None):
    """Ứng_viên = Mẫu_nền (lưới an toàn) ∪ Mẫu_sự_kiện (quanh đỉnh S(t)).
       sal_grid/sal_S: kết quả shot_saliency cho shot này (None -> chỉ dùng mẫu nền)."""
    s, e, fps = shot.start_frame, shot.end_frame, shot.fps
    n = e - s + 1
    dur = shot.duration_sec
    margin = int(CONFIG["boundary_margin_frac"] * n)
    lo, hi = (s + margin, e - margin) if n > 4 * margin + 2 else (s, e)

    def at(frac):  # frac trong [0,1] -> frame index
        return int(round(lo + frac * (hi - lo)))
    def clamp(j):
        return min(max(s, int(j)), e)

    # ---- Lớp 1: mẫu nền (lưới an toàn, giữ logic phân số cố định của pipeline cũ) ----
    if dur < 2.0:
        bg = [at(0.5)]
    elif dur < 8.0:
        bg = [at(0.25), at(0.5), at(0.75)]
    elif dur <= 20.0:
        bg = [at(p) for p in (0.1, 0.3, 0.5, 0.7, 0.9)]
    else:
        step = max(1, int(round(fps / CONFIG["candidate_fps"])))
        bg = list(range(lo, hi + 1, step))
    bg_u = sorted(set(clamp(j) for j in bg))

    # ---- Lớp 2: mẫu sự kiện quanh đỉnh S(t) (±δ) ----
    ev_best = {}
    if CONFIG["use_saliency"] and sal_grid is not None and sal_S is not None and len(sal_S):
        rad = max(1, int(round(CONFIG["saliency_peak_radius_sec"] * fps)))
        for pf, ps in _local_peaks(sal_grid, sal_S):
            for j in range(pf - rad, pf + rad + 1):
                if lo <= j <= hi:
                    jj = clamp(j)
                    if jj not in ev_best or ps > ev_best[jj]:
                        ev_best[jj] = ps
    ev_sorted = [j for j, _ in sorted(ev_best.items(), key=lambda kv: -kv[1])]  # S cao nhất trước

    cap = CONFIG["max_candidates_per_shot"]
    chosen, seen = [], set()
    def add(j):
        if j not in seen and len(chosen) < cap:
            seen.add(j); chosen.append(j)

    if CONFIG["candidate_priority"] == "event_first":
        # đúng report: ưu tiên mẫu sự kiện (S cao nhất), mẫu nền chỉ lấp phần còn lại
        for j in ev_sorted:
            add(j)
        remaining = cap - len(chosen)
        if remaining > 0:
            bg_left = [j for j in bg_u if j not in seen]
            for j in _even_subsample(bg_left, remaining):
                add(j)
    else:
        # balanced (mặc định): giữ lưới nền tối thiểu (an toàn) trước, phần còn lại ưu tiên sự kiện.
        # Dung hoà "ưu tiên sự kiện" và "không bỏ sót vùng thời gian" của report.
        if dur <= 20.0:
            bg_keep = bg_u                                  # nền vốn nhỏ (<=5) -> giữ hết
        else:
            reserve = max(1, int(round(cap * CONFIG["min_background_reserve_frac"])))
            bg_keep = _even_subsample(bg_u, min(len(bg_u), reserve))
        for j in bg_keep:
            add(j)
        for j in ev_sorted:                                 # phần còn lại: sự kiện theo S cao nhất
            add(j)
        for j in bg_u:                                      # còn dư ngân sách -> thêm nền cho phủ
            add(j)

    out = sorted(seen)
    return out or [int(round((s + e) / 2))]


## 7. Stability scoring (sharpness / brightness / contrast)

In [ ]:
def _norm_clip(x, lo, hi):
    return float(np.clip((x - lo) / (hi - lo + 1e-9), 0.0, 1.0))

def stability_scores(rgb: np.ndarray):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    # sharpness: Laplacian variance (log-scale để ổn định)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    sharp = _norm_clip(math.log1p(lap_var), math.log1p(5.0), math.log1p(1500.0))
    # brightness: FIX — Gaussian tâm 0.5, phạt theo mức độ lệch (không phạt nhị phân khi vượt ngưỡng).
    # Cảnh sáng tự nhiên (0.6–0.75) vẫn đạt điểm cao thay vì bị phạt oan như hàm lều cũ.
    mean_b = gray.mean() / 255.0
    sig = CONFIG["brightness_sigma"]
    bright = float(math.exp(-((mean_b - 0.5) ** 2) / (2.0 * sig * sig)))
    # contrast: std grayscale
    contrast = _norm_clip(gray.std() / 255.0, 0.04, 0.30)
    stab = (CONFIG["w_sharpness"] * sharp +
            CONFIG["w_brightness"] * bright +
            CONFIG["w_contrast"] * contrast)
    return sharp, bright, contrast, float(np.clip(stab, 0.0, 1.0))


## 8. Embedding — PE-Core-G14-448 (đa GPU không DataParallel)
PE-Core dùng RoPE có buffer ghim 1 device → **không thể bọc `nn.DataParallel`**. Thay vào đó: tạo **1 instance/GPU**, split batch chạy song song bằng thread. Có probe lúc build, lỗi multi-GPU thì **tự hạ về 1 GPU**. Fallback backend: PE → SigLIP2 (DataParallel ok) → handcrafted.

In [ ]:
import sys, threading

class _VisionWrap(nn.Module):   # chỉ dùng cho SigLIP2
    def __init__(self, model): super().__init__(); self.model = model
    def forward(self, pixel_values):
        return self.model.get_image_features(pixel_values=pixel_values)

class Embedder:
    def __init__(self):
        self.kind = None; self.dim = None; self.mode = None
        self.net = None; self.processor = None
        self.pe_models = []          # list[(device, model)]
        self.pe_transform = None
        self._build()

    # ---------- PE-Core ----------
    def _make_pe(self, pe, cfg, device):
        m = pe.CLIP.from_config(cfg, pretrained=True).eval()
        return m.half().to(device) if device != "cpu" else m.float()

    def _build_pe(self):
        repo = CONFIG["pe_repo_dir"]
        if repo and repo not in sys.path: sys.path.insert(0, repo)
        import core.vision_encoder.pe as pe
        import core.vision_encoder.transforms as PET
        cfg = CONFIG["pe_config"]
        if DEVICE != "cuda":
            devs = ["cpu"]
        elif CONFIG["pe_multi_gpu"] and N_GPU > 1:
            devs = [f"cuda:{i}" for i in range(N_GPU)]
        else:
            devs = ["cuda:0"]
        self.pe_models = []
        for i, d in enumerate(devs):
            m = self._make_pe(pe, cfg, d)
            if i == 0: self.pe_transform = PET.get_image_transform(m.image_size)
            self.pe_models.append((d, m))
        self.mode = "pe"
        # probe: nếu multi-GPU lỗi -> hạ về 1 GPU (cuda:0) cho chắc
        try:
            self.dim = self._encode_pe([np.zeros((8, 8, 3), np.uint8)] * max(2, len(self.pe_models))).shape[1]
        except Exception as e:
            if len(self.pe_models) > 1:
                log(f"[embed] PE multi-GPU lỗi ({type(e).__name__}: {e}); hạ về 1 GPU.")
                self.pe_models = [("cuda:0", self.pe_models[0][1].to("cuda:0"))]
                self.dim = self._encode_pe([np.zeros((8, 8, 3), np.uint8)]).shape[1]
            else:
                raise
        self.kind = f"pe:{cfg}" + (f"[{len(self.pe_models)}gpu]" if len(self.pe_models) > 1 else "")

    @torch.no_grad()
    def _encode_pe(self, images):
        ts = torch.stack([self.pe_transform(Image.fromarray(x)) for x in images])
        n = len(self.pe_models)
        if n == 1:
            dev, m = self.pe_models[0]
            x = ts.half().to(dev) if dev != "cpu" else ts.float()
            f = m.encode_image(x)
            return F.normalize(f.float(), dim=-1).cpu().numpy()
        chunks = list(torch.chunk(ts, n))
        outs = [None] * len(chunks)
        def run(i):
            dev, m = self.pe_models[i]
            with torch.no_grad():
                f = m.encode_image(chunks[i].half().to(dev))
            outs[i] = F.normalize(f.float(), dim=-1).cpu()
        th = [threading.Thread(target=run, args=(i,)) for i in range(len(chunks))]
        for t in th: t.start()
        for t in th: t.join()
        return torch.cat(outs, 0).numpy()

    # ---------- SigLIP2 (fallback) ----------
    def _build_siglip(self, src):
        from transformers import AutoModel, AutoProcessor
        self.processor = AutoProcessor.from_pretrained(src)
        model = AutoModel.from_pretrained(src, torch_dtype=torch.float32).eval()
        wrap = _VisionWrap(model)
        if DEVICE == "cuda":
            wrap = wrap.cuda()
            if N_GPU > 1: wrap = nn.DataParallel(wrap)
        self.net = wrap; self.mode = "siglip"; self.kind = f"siglip2:{src}"

    def _build(self):
        order = {"pe": ["pe", "siglip2", "handcrafted"],
                 "siglip2": ["siglip2", "handcrafted"],
                 "handcrafted": ["handcrafted"]}[CONFIG["embedder_backend"]]
        for b in order:
            try:
                if b == "pe":
                    self._build_pe()
                elif b == "siglip2":
                    self._build_siglip(CONFIG["local_model_path"] or CONFIG["model_name"])
                else:
                    self.kind = "handcrafted"; self.mode = "handcrafted"; self.net = None
                break
            except Exception as e:
                log(f"[embed] backend '{b}' fail -> next ({type(e).__name__}: {e})")
                self.net = None; self.mode = None; self.kind = None; self.pe_models = []
        if self.kind is None: self.kind = "handcrafted"; self.mode = "handcrafted"
        if self.mode == "handcrafted":
            log("LƯU Ý: handcrafted features -> nên đặt dbscan_eps ~0.35.")
        if self.dim is None:
            self.dim = self.encode([np.zeros((8, 8, 3), np.uint8)]).shape[1]
        log(f"Embedder = {self.kind} | dim={self.dim}")

    def _handcrafted(self, images):
        feats = []
        for rgb in images:
            im = cv2.resize(rgb, (64, 64))
            hsv = cv2.cvtColor(im, cv2.COLOR_RGB2HSV)
            h = cv2.calcHist([hsv], [0, 1], None, [16, 16], [0, 180, 0, 256]).flatten()
            gray = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY)
            gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0); gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1)
            edge = cv2.normalize(cv2.magnitude(gx, gy), None, 0, 1, cv2.NORM_MINMAX)
            ehist, _ = np.histogram(edge, bins=16, range=(0, 1))
            feats.append(np.concatenate([h, ehist.astype(np.float32)]))
        f = np.asarray(feats, np.float32)
        f /= (np.linalg.norm(f, axis=1, keepdims=True) + 1e-9)
        return f

    @torch.no_grad()
    def encode(self, images):
        if self.mode == "pe" and self.pe_models:
            out, per = [], CONFIG["embed_batch_size"] * max(1, len(self.pe_models))
            for i in range(0, len(images), per):
                out.append(self._encode_pe(images[i:i + per]))
            return np.concatenate(out, 0)
        if self.mode == "siglip" and self.net is not None:
            out, bs = [], CONFIG["embed_batch_size"] * max(1, N_GPU)
            for i in range(0, len(images), bs):
                chunk = [Image.fromarray(x) for x in images[i:i + bs]]
                px = self.processor(images=chunk, return_tensors="pt")["pixel_values"]
                if DEVICE == "cuda": px = px.to(DEVICE, non_blocking=True)
                feats = F.normalize(self.net(px).float(), dim=-1)
                out.append(feats.cpu().numpy())
            return np.concatenate(out, 0)
        return self._handcrafted(images)

EMBEDDER = Embedder()


## 9. DBSCAN clustering + chọn keyframe đại diện & ổn định (mỗi shot)

In [ ]:
def _max_keyframes_for_shot(dur):
    if dur < 4.0:  return 1
    if dur < 15.0: return 3
    return 5

def select_keyframes_for_shot(shot: Shot, cands):
    # cands: list[Candidate] thuộc shot (đã có embedding + stability).
    if not cands: return []
    cap = _max_keyframes_for_shot(shot.duration_sec)

    if len(cands) == 1:
        c = cands[0]; c.cluster_id = 0; c.representativeness = 1.0
        c.final_score = c.stability; c.selection_reason = "single_candidate"
        return [c]

    X = np.stack([c.embedding for c in cands]).astype(np.float32)
    labels = DBSCAN(eps=CONFIG["dbscan_eps"], min_samples=CONFIG["dbscan_min_samples"],
                    metric="cosine").fit_predict(X)

    chosen = []
    uniq = [l for l in sorted(set(labels)) if l != -1]

    if not uniq:  # toàn noise -> lấy frame stability cao nhất
        best = max(cands, key=lambda c: c.stability)
        best.cluster_id = -1; best.representativeness = 1.0
        best.final_score = best.stability; best.selection_reason = "all_noise_best_stability"
        return [best]

    for lab in uniq:
        members = [cands[i] for i in range(len(cands)) if labels[i] == lab]
        memb_emb = np.stack([m.embedding for m in members])
        centroid = memb_emb.mean(0); centroid /= (np.linalg.norm(centroid) + 1e-9)
        for m in members:
            m.representativeness = float(np.dot(m.embedding, centroid))
            m.final_score = (CONFIG["w_representativeness"] * m.representativeness +
                             CONFIG["w_stability"] * m.stability)
            m.cluster_id = int(lab)
        best = max(members, key=lambda m: m.final_score)
        best.selection_reason = "cluster_medoid_stable"
        chosen.append(best)

    # giữ thêm frame noise nếu shot dài & đủ đẹp & khác biệt rõ với cluster đã chọn
    if shot.duration_sec > 15.0:
        # FIX: cập nhật tập đại diện NGAY sau mỗi lần giữ -> điểm noise sau được so sánh
        # với điểm noise trước, tránh giữ 2 noise trùng nhau (trùng lặp ngầm).
        chosen_vecs = [c.embedding for c in chosen]
        for i in range(len(cands)):
            if labels[i] != -1: continue
            c = cands[i]
            if c.stability < CONFIG["noise_keep_min_stability"]: continue
            cur = np.stack(chosen_vecs)
            if float((cur @ c.embedding).max()) < (1 - CONFIG["dbscan_eps"]):
                c.cluster_id = -1; c.representativeness = float(c.stability)
                c.final_score = c.stability; c.selection_reason = "kept_distinct_noise"
                chosen.append(c); chosen_vecs.append(c.embedding)

    chosen.sort(key=lambda c: c.final_score, reverse=True)
    return chosen[:cap]


## 10. Deduplication — Module 7 (coverage suy giảm thời gian)
`Phủ(f,K) = max_k[ cos(e_f,e_k) · exp(−|t_f−t_k|/τ) ]`, giữ `f` nếu `Phủ < θ`. Frame nét (stability cao) làm neo trước. pHash chỉ còn là bộ lọc tăng tốc, không tham gia quyết định. → bảo toàn các lần xuất hiện độc lập cách xa thời gian (tốt cho recall AIC).

In [ ]:
def dedup_keyframes(selected):
    """Loại trùng bằng ĐỘ PHỦ SUY GIẢM THỜI GIAN (thay ngưỡng cứng cosine/pHash cũ):
         Phủ(f,K) = max_{k∈K}[ cos(e_f,e_k) · exp(-|t_f−t_k|/τ) ]
       Giữ f nếu Phủ(f,K) < θ; loại nếu ≥ θ. Hai frame giống nhau nhưng CÁCH XA thời gian
       (>5 phút) gần như luôn được giữ -> bảo toàn các lần xuất hiện độc lập (tốt cho recall).
       Xử lý theo stability giảm dần: frame nét làm neo trước.
       pHash chỉ còn là bộ lọc tăng tốc: pHash quá khác -> chắc chắn không trùng -> bỏ qua cosine."""
    if not selected: return []
    tau = max(1e-6, CONFIG["dedup_tau_sec"])
    theta = CONFIG["dedup_coverage_threshold"]
    ph_pre = CONFIG["dedup_phash_prefilter"]
    for c in selected:
        if c.phash is None:
            c.phash = imagehash.phash(Image.fromarray(c.image_rgb))
    order = sorted(selected, key=lambda c: c.stability, reverse=True)  # neo chất lượng cao trước
    keep = []
    for c in order:
        covered = False
        for k in keep:
            if (c.phash - k.phash) > ph_pre:      # cấu trúc khác hẳn -> không trùng -> bỏ qua cosine
                continue
            cos = float(np.dot(c.embedding, k.embedding))
            cov = cos * math.exp(-abs(c.pts_time - k.pts_time) / tau)
            if cov >= theta:
                covered = True; break
        if not covered:
            keep.append(c)
    return sorted(keep, key=lambda c: c.frame_idx)


## 11. Pipeline 1 video → ghi đúng format AIC

In [ ]:
def video_id_from_path(path):
    return Path(path).stem   # K20_V001.mp4 -> K20_V001 (đã global-unique)

def already_done(video_id):
    csv = OUT / "map-keyframes" / f"{video_id}.csv"
    kfd = OUT / "keyframes" / video_id
    return csv.exists() and kfd.exists() and any(kfd.glob("*.jpg"))

def extract_one_video(path, verbose=False):
    vid = video_id_from_path(path)
    if CONFIG["skip_existing"] and already_done(vid):
        return {"video_id": vid, "status": "skip", "n_keyframes": None}

    fps, total, w, h = probe_video(path)
    shots, used = detect_shots(path, fps, total)

    # tín hiệu nổi bật S(t) toàn video (1 lượt decode low-res + |d/dt RMS| audio)
    if CONFIG["use_saliency"]:
        sal_grid, sal_dvis, sal_daud = compute_saliency(path, fps, total)
    else:
        sal_grid, sal_dvis, sal_daud = np.array([], int), np.array([], np.float32), None
    has_audio = sal_daud is not None
    if verbose:
        log(f"{vid}: fps={fps:.2f} total={total} shots={len(shots)} ({used}) | "
            f"saliency={'on' if CONFIG['use_saliency'] else 'off'} audio={'yes' if has_audio else 'no'}")

    # 1) gom candidate index theo từng shot (lái bởi S(t) per-shot) -> đọc frame 1 lượt
    shot_cand_idx = {}
    for sh in shots:
        if CONFIG["use_saliency"] and sal_grid.size:
            g, S = shot_saliency(sh, sal_grid, sal_dvis, sal_daud)
            shot_cand_idx[sh.shot_id] = candidate_indices(sh, g, S)
        else:
            shot_cand_idx[sh.shot_id] = candidate_indices(sh)
    all_idx = sorted({i for v in shot_cand_idx.values() for i in v})
    frames = read_frames_at_indices(path, all_idx, CONFIG["save_max_side"])
    if not frames:
        return {"video_id": vid, "status": "no_frames", "n_keyframes": 0}

    # 2) build candidates + stability
    cands_by_shot = {sh.shot_id: [] for sh in shots}
    flat = []
    for sh in shots:
        for fi in shot_cand_idx[sh.shot_id]:
            if fi not in frames: continue
            rgb = frames[fi]
            shp, brt, con, stab = stability_scores(rgb)
            c = Candidate(sh.shot_id, fi, fi / fps, rgb, shp, brt, con, stab)
            cands_by_shot[sh.shot_id].append(c); flat.append(c)
    if not flat:
        return {"video_id": vid, "status": "no_candidates", "n_keyframes": 0}

    # 3) embed tất cả candidate 1 lần (batch, dùng cả 2 GPU)
    embs = EMBEDDER.encode([c.image_rgb for c in flat])
    for c, e in zip(flat, embs): c.embedding = e

    # 4) DBSCAN + chọn theo từng shot
    selected = []
    for sh in shots:
        selected += select_keyframes_for_shot(sh, cands_by_shot[sh.shot_id])

    # 5) dedup toàn video
    final = dedup_keyframes(selected)

    # 6) ghi output đúng format AIC
    kf_dir = OUT / "keyframes" / vid
    kf_dir.mkdir(parents=True, exist_ok=True)
    for f in kf_dir.glob("*.jpg"): f.unlink()   # ghi lại sạch nếu chạy đè
    rows = []
    for n, c in enumerate(final, start=1):
        fname = f"{n:03d}.jpg"
        Image.fromarray(c.image_rgb).save(kf_dir / fname, quality=CONFIG["jpg_quality"])
        rows.append({
            "n": n, "frame_idx": int(c.frame_idx), "pts_time": round(c.pts_time, 3),
            "fps": round(fps, 4), "timestamp": sec_to_hms(c.pts_time),
            "shot_id": int(c.shot_id), "cluster_id": int(c.cluster_id),
            "stability_score": round(c.stability, 4), "sharpness_score": round(c.sharpness, 4),
            "brightness_score": round(c.brightness, 4), "contrast_score": round(c.contrast, 4),
            "representativeness_score": round(c.representativeness, 4),
            "selection_reason": c.selection_reason,
            "keyframe_id": f"{vid}#{n:03d}", "image_path": str(kf_dir / fname),
        })
    df = pd.DataFrame(rows)
    df.to_csv(OUT / "map-keyframes" / f"{vid}.csv", index=False)

    # lưu embedding (hàng i ↔ n=i+1) để tái dùng cho bước index, khỏi encode lại
    if CONFIG["save_embeddings"] and final:
        emb_dir = OUT / CONFIG["embeddings_dir"]; emb_dir.mkdir(parents=True, exist_ok=True)
        arr = np.stack([c.embedding for c in final]).astype(np.float16)
        np.save(emb_dir / f"{vid}.npy", arr)

    return {"video_id": vid, "status": "ok", "n_keyframes": len(final),
            "n_shots": len(shots), "shot_method": used, "has_audio": has_audio}


## 12. Tìm video & batch runner (chọn loại video + resume + chia ca)
`discover_videos()` lọc theo `CONFIG["video_groups"]` (prefix của video_id, vd `["K01","L21"]`; rỗng = tất cả), rồi `RANGE_START`/`RANGE_END` cắt ca. Dùng `list_groups()` để xem các nhóm có trong input trước khi đặt bộ lọc.

In [ ]:
def _match_group(video_id):
    """Khớp video_id với CONFIG['video_groups'] theo prefix (không phân biệt hoa thường).
       [] / None -> nhận tất cả. Vd groups=['K01','L21'] khớp 'K01_V003', 'L21_V010'."""
    groups = CONFIG.get("video_groups") or []
    if not groups:
        return True
    vid = video_id.upper()
    return any(vid.startswith(str(g).upper()) for g in groups)

def discover_videos():
    paths = []
    for root in CONFIG["input_roots"]:
        for ext in CONFIG["video_exts"]:
            paths += glob.glob(os.path.join(root, "**", f"*{ext}"), recursive=True)
    # unique theo video_id (phòng trùng), lọc theo nhóm, sort theo tên
    seen, uniq = set(), []
    for p in sorted(paths, key=lambda x: Path(x).stem):
        vid = Path(p).stem
        if vid in seen: continue
        if not _match_group(vid): continue        # <-- chọn loại video (K01, K02, L21, L22, ...)
        seen.add(vid); uniq.append(p)
    return uniq

def list_groups():
    """Liệt kê các nhóm video (prefix trước '_') có trong input — tiện để biết đặt video_groups gì."""
    bak = CONFIG["video_groups"]; CONFIG["video_groups"] = []
    allv = discover_videos(); CONFIG["video_groups"] = bak
    from collections import Counter
    cnt = Counter((Path(p).stem.split("_")[0]) for p in allv)
    for g, c in sorted(cnt.items()):
        print(f"  {g:<8} {c} video")
    return cnt

def run_batch():
    vids = discover_videos()
    groups = CONFIG.get("video_groups") or []
    a = CONFIG["RANGE_START"]; b = CONFIG["RANGE_END"] if CONFIG["RANGE_END"] is not None else len(vids)
    subset = vids[a:b]
    gtxt = f"groups={groups}" if groups else "groups=ALL"
    log(f"{gtxt} | {len(vids)} video khớp | chạy ca [{a}:{b}] = {len(subset)} video")
    summary = []
    t0 = time.time()
    for p in tqdm(subset, desc="videos"):
        try:
            summary.append(extract_one_video(p))
        except Exception as e:
            log(f"LỖI {Path(p).stem}: {e}")
            traceback.print_exc()
            summary.append({"video_id": Path(p).stem, "status": f"error:{type(e).__name__}", "n_keyframes": 0})
    dt = time.time() - t0
    sdf = pd.DataFrame(summary)
    sdf.to_csv(OUT / f"run_summary_{a}_{b}.csv", index=False)
    ok = (sdf["status"] == "ok").sum() if len(sdf) else 0
    log(f"Xong {len(subset)} video trong {dt/60:.1f} phút | ok={ok} | "
        f"keyframes={int(sdf.loc[sdf.status=='ok','n_keyframes'].sum()) if ok else 0}")
    return sdf


## 13. Smoke test — chạy 1 video kiểm tra trước

In [ ]:
# Các nhóm video có trong input (đặt CONFIG["video_groups"] để chỉ xử lý nhóm mong muốn)
print("Nhóm video trong input:")
try:
    list_groups()
except Exception as _e:
    print("  (chưa liệt kê được:", _e, ")")

_vids = discover_videos()
print("\nSố video khớp bộ lọc:", len(_vids), "| video_groups =", CONFIG["video_groups"] or "ALL")
print("5 video đầu:", [Path(v).stem for v in _vids[:5]])

if _vids:
    _bak = CONFIG["skip_existing"]
    CONFIG["skip_existing"] = False           # ép chạy để test
    res = extract_one_video(_vids[0], verbose=True)
    CONFIG["skip_existing"] = _bak
    print(res)
    _vid = res["video_id"]
    display(pd.read_csv(OUT / "map-keyframes" / f"{_vid}.csv").head(12))
else:
    print("⚠️ Không có video khớp — kiểm tra đã Attach dataset và CONFIG['video_groups'] đúng chưa.")


## 14. Preview keyframes vừa trích

In [ ]:
import matplotlib.pyplot as plt

def preview(video_id, max_images=12):
    csv = OUT / "map-keyframes" / f"{video_id}.csv"
    if not csv.exists():
        print("Không có map cho", video_id); return
    df = pd.read_csv(csv).head(max_images)
    cols = 4; rows = int(np.ceil(len(df) / cols))
    plt.figure(figsize=(cols * 3, rows * 2.6))
    for i, r in df.iterrows():
        plt.subplot(rows, cols, i + 1)
        plt.imshow(Image.open(r["image_path"])); plt.axis("off")
        plt.title(f"n{int(r['n'])} f{int(r['frame_idx'])}\n{r['timestamp']} s={r['stability_score']:.2f}",
                  fontsize=8)
    plt.tight_layout(); plt.show()

if _vids:
    preview(res["video_id"])


## 14c. Preview TOÀN BỘ keyframe (contact-sheet)
Ghép tất cả keyframe của 1 video vào ảnh lưới (nhẹ hơn nhiều so với 294 subplot matplotlib), tự chia trang và lưu PNG ra `/kaggle/working/preview_<video_id>.png` để tải về. Nhãn mỗi ô: `#n  timestamp  shot_id  stability`.

In [ ]:
import math
from PIL import Image, ImageDraw

def preview_all(video_id=None, cols=8, thumb_w=200, pad=4, label=True,
                per_page=160, save=True, bg=(18, 18, 18), fg=(240, 240, 240)):
    """Ghép TẤT CẢ keyframe của 1 video thành contact-sheet (nhẹ, 1 ảnh/trang) + lưu PNG để tải về.
       - video_id=None -> dùng video smoke-test (biến `res`), không có thì map-keyframes đầu tiên.
       - cols: số cột | thumb_w: bề ngang thumbnail | per_page=None -> dồn 1 trang duy nhất."""
    if video_id is None:
        try:
            video_id = res["video_id"]
        except Exception:
            maps = sorted((OUT / "map-keyframes").glob("*.csv"))
            if not maps:
                print("Chưa có map-keyframes nào."); return
            video_id = maps[0].stem
    csv = OUT / "map-keyframes" / f"{video_id}.csv"
    if not csv.exists():
        print("Không có map cho", video_id); return
    df = pd.read_csv(csv)
    if df.empty:
        print(video_id, "— 0 keyframe."); return

    rows = df.to_dict("records"); n = len(rows)
    thumb_h = int(round(thumb_w * 9 / 16)); lab_h = 16 if label else 0
    cell_w, cell_h = thumb_w + pad, thumb_h + lab_h + pad
    per_page = per_page or n; n_pages = math.ceil(n / per_page)
    print(f"{video_id}: {n} keyframe | {cols} cột | {n_pages} trang")

    out = []
    for pg in range(n_pages):
        chunk = rows[pg * per_page:(pg + 1) * per_page]
        nrows = math.ceil(len(chunk) / cols)
        canvas = Image.new("RGB", (cols * cell_w + pad, nrows * cell_h + pad), bg)
        draw = ImageDraw.Draw(canvas)
        for i, r in enumerate(chunk):
            cr, cc = divmod(i, cols); x = pad + cc * cell_w; y = pad + cr * cell_h
            try:
                im = Image.open(r["image_path"]).convert("RGB")
            except Exception:
                im = Image.new("RGB", (thumb_w, thumb_h), (60, 0, 0))
            im.thumbnail((thumb_w, thumb_h))
            box = Image.new("RGB", (thumb_w, thumb_h), bg)       # letterbox vào ô đều nhau
            box.paste(im, ((thumb_w - im.width) // 2, (thumb_h - im.height) // 2))
            canvas.paste(box, (x, y))
            if label:
                txt = f"#{int(r['n'])} {str(r['timestamp'])[:8]} s{int(r['shot_id'])} st{r['stability_score']:.2f}"
                draw.rectangle([x, y + thumb_h, x + thumb_w, y + thumb_h + lab_h], fill=(0, 0, 0))
                draw.text((x + 2, y + thumb_h + 2), txt, fill=fg)
        out.append(canvas)
        if save:
            sfx = "" if n_pages == 1 else f"_p{pg + 1:02d}"
            outp = OUT / f"preview_{video_id}{sfx}.png"
            canvas.save(outp); print("  saved:", outp)
        display(canvas)
    return out

# ---- DÙNG ----
preview_all()                                          # toàn bộ keyframe video smoke-test
# preview_all("K01_V001", cols=10)                     # video cụ thể, 10 cột
# preview_all("K01_V001", per_page=None, thumb_w=150)  # dồn 1 trang, thumbnail nhỏ


## 14b. Auto-tune `dbscan_eps` (khỏi mò)
eps trong không gian cosine = `1 − ngưỡng-similarity-gộp`. Cell này chạy trên vài video mẫu, vẽ **đường k-distance** (k=`min_samples`) trong từng shot và lấy **knee** làm eps gợi ý, đồng thời **sweep** một dải eps để xem số keyframe/shot và coverage. Chọn xong set 1 dòng `CONFIG["dbscan_eps"]=...` rồi mới chạy batch.

*Coverage* = trung bình (similarity cao nhất của mỗi candidate tới tập keyframe đã chọn). eps nhỏ → nhiều keyframe → coverage cao nhưng dễ dư. Retrieval ưu tiên recall → lấy eps hơi nhỏ hơn knee (~0.02–0.03).

In [ ]:
def _knee_point(y_sorted):
    # kneedle-style: điểm xa nhất tới dây cung nối đầu-cuối của đường cong lồi tăng dần
    n = len(y_sorted)
    if n < 3: return None
    x = np.arange(n, dtype=float); y = np.asarray(y_sorted, float)
    denom = np.hypot(x[-1] - x[0], y[-1] - y[0]) + 1e-12
    d = np.abs((y[-1] - y[0]) * (x - x[0]) - (x[-1] - x[0]) * (y - y[0])) / denom
    return int(np.argmax(d))

def autotune_eps(sample_videos,
                 eps_grid=(0.10, 0.12, 0.14, 0.16, 0.18, 0.20, 0.22, 0.25, 0.30),
                 plot=True):
    k = CONFIG["dbscan_min_samples"]
    shot_mats = []   # (shot, embeddings) cho từng shot có >=2 candidate
    for vp in sample_videos:
        fps, total, _, _ = probe_video(vp)
        shots, _ = detect_shots(vp, fps, total)
        idx = {sh.shot_id: candidate_indices(sh) for sh in shots}
        allidx = sorted({i for v in idx.values() for i in v})
        frames = read_frames_at_indices(vp, allidx, CONFIG["save_max_side"])
        for sh in shots:
            ims = [frames[i] for i in idx[sh.shot_id] if i in frames]
            if len(ims) >= 2:
                shot_mats.append((sh, EMBEDDER.encode(ims)))
    if not shot_mats:
        print("Không đủ shot/candidate để tune."); return CONFIG["dbscan_eps"], []

    # ---- k-distance (trong từng shot) ----
    kd = []
    for sh, X in shot_mats:
        if len(X) <= k: continue
        D = 1.0 - X @ X.T
        np.fill_diagonal(D, 2.0)
        kd.extend(np.sort(D, axis=1)[:, k - 1].tolist())
    kd = np.sort(np.array(kd)) if kd else np.array([CONFIG["dbscan_eps"]])
    knee = _knee_point(kd)
    eps_knee = float(kd[knee]) if (knee is not None) else float(np.percentile(kd, 80))

    if plot:
        try:
            import matplotlib.pyplot as plt
            plt.figure(figsize=(6, 3))
            plt.plot(kd, lw=1.5)
            if knee is not None: plt.axhline(eps_knee, color="r", ls="--",
                                             label=f"knee eps={eps_knee:.3f}")
            plt.xlabel(f"điểm (sorted) | {len(shot_mats)} shots"); plt.ylabel(f"k-dist (k={k}) cosine")
            plt.title("k-distance graph"); plt.legend(); plt.tight_layout(); plt.show()
        except Exception:
            pass

    # ---- sweep eps: keyframes/shot + coverage ----
    print(f"\n{'eps':>5} {'kf/shot med':>11} {'kf/shot mean':>12} {'coverage':>9}")
    rows = []
    old = CONFIG["dbscan_eps"]
    for eps in eps_grid:
        CONFIG["dbscan_eps"] = eps
        kfs, covs = [], []
        for sh, X in shot_mats:
            cands = []
            for e in X:
                c = Candidate(sh.shot_id, 0, 0.0, np.zeros((2, 2, 3), np.uint8))
                c.embedding = e; c.stability = 0.7
                cands.append(c)
            sel = select_keyframes_for_shot(sh, cands)
            kfs.append(len(sel))
            sel_emb = np.stack([c.embedding for c in sel])
            covs.append(float((X @ sel_emb.T).max(axis=1).mean()))
        med, mean, cov = float(np.median(kfs)), float(np.mean(kfs)), float(np.mean(covs))
        rows.append((eps, med, mean, cov))
        print(f"{eps:>5.2f} {med:>11.1f} {mean:>12.2f} {cov:>9.3f}")
    CONFIG["dbscan_eps"] = old
    print(f"\n→ eps gợi ý (knee k-distance): {eps_knee:.3f}")
    print("  Áp dụng:  CONFIG['dbscan_eps'] = round(%.3f, 3)   # hoặc nhỏ hơn ~0.02–0.03 cho recall" % eps_knee)
    return eps_knee, rows

if _vids:
    eps_suggest, _ = autotune_eps(_vids[:5])
    # CONFIG["dbscan_eps"] = round(eps_suggest, 3)   # bỏ comment để áp dụng


## 15. Chạy toàn bộ (batch)
Bật `skip_existing` để resume. Chia ca giữa các session bằng `RANGE_START`/`RANGE_END` ở CONFIG.

In [ ]:
# CONFIG["RANGE_START"] = 0
# CONFIG["RANGE_END"]   = 300     # vd ca 1: 0-300, ca 2: 300-600, ...
summary_df = run_batch()
summary_df["status"].value_counts()


## 16. (Tuỳ chọn) Gộp metadata & đóng gói
Với AIC nhiều nghìn video, **không nên zip toàn bộ** (vượt giới hạn output/time). Cứ để `keyframes/` + `map-keyframes/` làm **Output dataset** của Kaggle để pipeline embedding/index dùng tiếp.
Cell dưới chỉ gộp toàn bộ map-keyframes thành 1 file để tiện kiểm tra.

In [ ]:
maps = sorted((OUT / "map-keyframes").glob("*.csv"))
if maps:
    big = []
    for m in maps:
        d = pd.read_csv(m); d.insert(0, "video_id", m.stem); big.append(d)
    alldf = pd.concat(big, ignore_index=True)
    alldf.to_csv(OUT / "all_keyframes.csv", index=False)
    print("Tổng keyframe:", len(alldf), "| video:", alldf['video_id'].nunique())
    display(alldf.head())
else:
    print("Chưa có map-keyframes nào.")
